# Stage 10/11 (repurposed) — General F&O trend-following ranking

`score_for_trend_following()` in `strategy_scoring.py` — ranks the
full F&O stock+index 1D universe (see `trend_universe.py` and README
Stage 10 for the repurposing history) for what it's actually for: buy the
share on a fresh UP flip, exit on the opposite DOWN flip, pure cash equity.
Different from the other four scorers in this project — favors a large
favorable move on the UP flips specifically (buying the underlying has no
structural cap the way credit spreads/short IC's containment objective or
debit spread/option buying's ATR-bounded structure do). No stop-loss:
`src/stop_loss_test.py` found no candidate stop rule beats holding to the
natural DOWN-flip exit, so this scores the actual strategy, not a
hypothetical stopped-out version of it.

`security_type` ("Stock"/"Index") is retained purely as a display/filter
column — the ranking is one table across the whole universe, not split by
type (that split existed when the universe included ETFs, which had
structurally smaller typical moves; there are no ETFs in the universe now).

Requires notebook 10 to have been run with real deep history first — if
`up_closed_count` looks thin across the board, that's the symptom of the
now-fixed regression documented in README Stage 10/11 — re-run notebook 10.

In [1]:
import sys
sys.path.append("..")

from src.strategy_scoring import score_for_trend_following, save_ranking

ranked, watchlist = score_for_trend_following()
print(f"Ranked: {len(ranked)}, Watchlist: {len(watchlist)}")
ranked["security_type"].value_counts()

Ranked: 205, Watchlist: 19


security_type
Stock    199
Index      6
Name: count, dtype: int64

In [2]:
# Notebook 13 (live screener) reads these two files to merge trend_score
# into its status table. This save step was accidentally dropped when the
# pledged-1D notebooks were repurposed (commit cb9dbea) - the old notebook
# saved RANKING_/WATCHLIST_pledged_trend_following.xlsx right here.
save_ranking(ranked, "RANKING_trend_following.xlsx")
save_ranking(watchlist, "WATCHLIST_trend_following.xlsx")

Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/RANKING_trend_following.xlsx
Saved ranking to /Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_trend_following.xlsx


'/Users/hgmac/Documents/3_trading/Quant/exports/WATCHLIST_trend_following.xlsx'

## Ranked universe

In [3]:
ranked.reset_index(drop=True)

,symbol,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,FINNIFTY,Index,93.2,,,47,55.3,2.14,0.54,2.47
1,MIDCPNIFTY,Index,91.6,,,92,51.1,1.22,0.08,1.64
2,TITAN,Stock,91.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 4.65pp),,36,61.1,6.70,2.05,4.24
3,BOSCHLTD,Stock,89.5,,,32,56.2,4.57,1.64,5.68
4,BAJFINANCE,Stock,87.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 5.58pp),,39,59.0,7.43,1.85,4.36
...,...,...,...,...,...,...,...,...,...,...
200,MPHASIS,Stock,6.7,⚠ RELIES ON OUTLIER WINS (avg-median gap 16.42pp),,30,30.0,11.16,-5.26,8.20
201,CDSL,Stock,6.3,⚠ RELIES ON OUTLIER WINS (avg-median gap 14.27pp),,29,31.0,8.32,-5.95,8.97
202,IDEA,Stock,5.5,⚠ RELIES ON OUTLIER WINS (avg-median gap 10.32pp),,39,30.8,0.20,-10.12,12.06
203,JSWENERGY,Stock,4.9,⚠ RELIES ON OUTLIER WINS (avg-median gap 16.46pp),,38,26.3,11.30,-5.16,8.77


In [4]:
ranked[ranked["security_type"] == "ETF"].reset_index(drop=True)

,symbol,security_type,trend_score,outlier_reliance_flag,gap_risk_flag,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up


In [5]:
# Instruments below the 15 UP-closed-flip floor - not blended into the
# ranking, but not dropped either. Re-check as more calendar time passes.
print(f"On watchlist: {len(watchlist)}")
watchlist.sort_values("avg_pct_move_up", ascending=False).reset_index(drop=True)

On watchlist: 19


,symbol,security_type,up_closed_count,win_rate_up_pct,avg_pct_move_up,median_pct_move_up,median_max_adverse_up
0,ATHERENERG,Stock,3,66.7,65.62,97.52,6.53
1,ANANDRATHI,Stock,13,38.5,29.81,-0.01,4.55
2,KAYNES,Stock,11,63.6,22.09,5.46,9.78
3,ETERNAL,Stock,11,54.5,20.26,3.05,7.55
4,DATAPATTNS,Stock,14,57.1,13.26,1.64,9.82
5,SYRMA,Stock,13,53.8,10.46,0.18,10.71
6,HYUNDAI,Stock,4,50.0,9.40,1.77,7.92
7,KFINTECH,Stock,12,33.3,5.28,-2.81,6.76
8,DELHIVERY,Stock,11,36.4,5.24,-1.07,5.10
9,IREDA,Stock,10,30.0,3.59,-3.06,7.30
